# NB17B · Auditoría de particiones (versión corregida, v2)

**AgriEngineering 4664292.** Verifica los conjuntos de calibración/test congelados de NB12 y tres subparticiones independientes por `duplicate_group_id`. **No reentrena modelos ni cambia el test.**

**Correcciones v2:** importación explícita de `json`; localización directa del ZIP NB12 sin búsqueda recursiva de toda Mi unidad; ordenación por `record_id` para reproducir el mismo orden de NB17; diagnóstico explícito si faltan dependencias/archivos. La última celda guarda en Drive y descarga un ZIP.

**Uso:** Google Colab → CPU → Ejecutar todas. Elige la cuenta de Drive que contiene `Grain_Robustness_Q1`. Si algún veredicto es `FAIL`, no integrar conclusiones conformales al manuscrito.


In [ ]:
import sys, subprocess, importlib.util, json, io, zipfile, hashlib
from pathlib import Path
from datetime import datetime, timezone

required = {'pyarrow':'pyarrow', 'pandas':'pandas', 'scikit-learn':'sklearn'}
missing = [pip for pip,mod in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from IPython.display import display
from google.colab import drive

MYDRIVE = Path('/content/drive/MyDrive')
if not MYDRIVE.is_dir():
    drive.mount('/content/drive')
if not MYDRIVE.is_dir():
    raise FileNotFoundError('No se pudo montar /content/drive/MyDrive. Autoriza Google Drive en Colab.')

PROJECT = MYDRIVE / 'Grain_Robustness_Q1'
if not PROJECT.is_dir():
    raise FileNotFoundError(f'Carpeta de proyecto no encontrada: {PROJECT}. Verifica la cuenta conectada.')

SRC_NAME = 'NOTEBOOK12_REVIEWER_DRIVEN_REANALYSIS_OUTPUTS_V2.zip'
NB12 = PROJECT / SRC_NAME
if not NB12.is_file():
    raise FileNotFoundError(f'Falta ZIP NB12 en {NB12}. No se cambia ni se reconstruye la fuente.')

EXPECTED_SHA = '68eee4090933b5fb31e611f8175c773ee3395af979e3729db68af61b06da05fa'
source_sha = hashlib.sha256(NB12.read_bytes()).hexdigest()
if source_sha != EXPECTED_SHA:
    raise AssertionError(f'ZIP NB12 no coincide con el SHA-256 congelado: {source_sha}')
print('✅ Fuente NB12 verificada:', NB12)
print('✅ SHA-256:', source_sha)


In [ ]:
REPS = {'dry_bean':'R1_full_native', 'iniap_rice':'R4_complete_log_area'}
SEED = 20261027
results = []

with zipfile.ZipFile(NB12, 'r') as z:
    if z.testzip() is not None:
        raise AssertionError('ZIP NB12 dañado: fallo de integridad.')
    members = z.namelist()

    def read_orig(dataset, rep, split):
        suffix = f'/{dataset}/{rep}/rbf_svm_{split}_predictions.parquet'
        matches = [m for m in members if m.endswith(suffix)]
        if len(matches) != 1:
            raise FileNotFoundError(f'Se esperaba un Parquet único para {suffix}; se hallaron {len(matches)}')
        df = pd.read_parquet(io.BytesIO(z.read(matches[0])))
        for col in ('record_id', 'duplicate_group_id', 'class_label'):
            if col not in df.columns:
                raise KeyError(f'Falta {col} en {matches[0]}')
            df[col] = df[col].astype(str)
        # Reproduce estrictamente el orden utilizado por load_pred() del NB17.
        df = df.sort_values('record_id').reset_index(drop=True)
        if not df.record_id.is_unique:
            raise AssertionError(f'record_id repetido: {dataset}/{split}')
        if (df.groupby('duplicate_group_id').class_label.nunique() > 1).any():
            raise AssertionError(f'Grupos de duplicados con más de una clase: {dataset}/{split}')
        return df

    for ds, rep in REPS.items():
        cal = read_orig(ds, rep, 'calibration')
        test = read_orig(ds, rep, 'test')
        ids_cal, ids_test = set(cal.record_id), set(test.record_id)
        gs_cal, gs_test = set(cal.duplicate_group_id), set(test.duplicate_group_id)
        overlap_id = ids_cal & ids_test
        overlap_group = gs_cal & gs_test

        groups = cal[['duplicate_group_id', 'class_label']].drop_duplicates('duplicate_group_id').copy()
        g_ids, g_y = groups.duplicate_group_id.to_numpy(), groups.class_label.to_numpy()
        if any((groups.class_label == label).sum() < 3 for label in groups.class_label.unique()):
            raise AssertionError(f'Muy pocos grupos por clase para la estratificación: {ds}')
        i_temp, i_rest = train_test_split(np.arange(len(groups)), train_size=.4,
                                           random_state=SEED, stratify=g_y)
        i_sel, i_conf = train_test_split(i_rest, train_size=.5,
                                         random_state=SEED+1, stratify=g_y[i_rest])
        parts = [set(g_ids[idx]) for idx in (i_temp, i_sel, i_conf)]
        inner_overlap = sum(len(parts[i] & parts[j]) for i,j in ((0,1),(0,2),(1,2)))
        if set.union(*parts) != gs_cal:
            raise AssertionError(f'La subdivisión de {ds} no cubre todos los grupos de calibración.')

        passed = (len(overlap_id) == 0 and len(overlap_group) == 0 and inner_overlap == 0)
        results.append({
            'dataset':ds, 'representation':rep,
            'n_cal_rows':len(cal), 'n_cal_groups':len(gs_cal),
            'n_test_rows':len(test), 'n_test_groups':len(gs_test),
            'overlap_record_ids':len(overlap_id),
            'overlap_duplicate_groups':len(overlap_group),
            'overlap_calibration_subgroups':inner_overlap,
            'temperature_groups':len(parts[0]),
            'selective_groups':len(parts[1]),
            'conformal_groups':len(parts[2]),
            'verdict':'PASS' if passed else 'FAIL'
        })
        print(f"{ds}: {'PASS' if passed else 'FAIL'} | solapamiento ID={len(overlap_id)}, "
              f"grupos={len(overlap_group)}, subparticiones={inner_overlap}")

report = pd.DataFrame(results)
display(report)
if (report.verdict == 'PASS').all():
    print('✅ Independencia de identificadores/grupos confirmada para ambas fuentes NB12.')
else:
    print('⚠️ FAIL: guardar el ZIP para auditoría; NO utilizar coberturas en manuscrito antes de resolverlo.')


## Interpretación

Un `PASS` en los dos conjuntos cierra el control **calibración/test** a nivel de identificadores bajo las fuentes NB12, y prueba también que las tres partes internas de calibración son disjuntas por grupo. Adjuntar el ZIP de auditoría al paquete de evidencia del artículo. No inferir de esta prueba independencia entre campañas, ausencia de correlación biológica o validez fuera de distribución.

## Exportación a My Drive y descarga de ZIP para revisión

Esta celda guarda **todos los resultados de NB17B** en la carpeta del proyecto y crea un solo ZIP con el CSV de auditoría, metadatos, nota metodológica y hashes SHA-256. Descarga automáticamente el ZIP en Google Colab para enviarlo por el chat. **No incluye archivos originales ni modifica NB12, NB16 o NB17.**


In [ ]:
from datetime import datetime, timezone
import json, hashlib, zipfile
from google.colab import files as colab_files

REVIEW = PROJECT / 'Revision_4664292_Ensayos_2026'
OUT_B = REVIEW / 'resultados_colab' / 'NB17B_4664292'
OUT_B.mkdir(parents=True, exist_ok=True)
# Eliminar únicamente resultados NB17B anteriores; jamás modificar fuentes NB12/NB16/NB17.
for stale in OUT_B.glob('NB17B_*'):
    if stale.is_file():
        stale.unlink()

# Guardar evidencia incluso si el control detecta solapamientos:
report_file = OUT_B / 'NB17B_01_control_particiones.csv'
report.to_csv(report_file, index=False)

meta = {
    'manuscript': 'agriengineering-4664292',
    'notebook': 'NB17B_Auditoria_Particiones_4664292',
    'execution_utc': datetime.now(timezone.utc).isoformat(),
    'source_nb12_filename': NB12.name,
    'source_nb12_sha256': source_sha,
    'seed': SEED,
    'datasets': sorted(REPS),
    'verdict_by_dataset': dict(zip(report['dataset'], report['verdict'])),
    'all_pass': bool((report['verdict'] == 'PASS').all()),
    'note': 'Solo auditoria de particiones. No reentrenamiento ni modificacion de fuentes congeladas.'
}
(OUT_B / 'NB17B_02_metadatos.json').write_text(
    json.dumps(meta, ensure_ascii=False, indent=2), encoding='utf-8')
(OUT_B / 'NB17B_03_LEEME.txt').write_text(
    'AgriEngineering 4664292 | Auditoria NB17B\n'
    'El CSV informa solapamiento por record_id, duplicate_group_id y las tres subparticiones.\n'
    'PASS no demuestra ausencia de cualquier otra fuga ni validez fuera de distribucion.\n'
    'Si hay FAIL, suspender la interpretacion conformal y revisar las fuentes.\n',
    encoding='utf-8')

# Manifiesto de los archivos, independiente del contenido del ZIP:
inputs = sorted(p for p in OUT_B.glob('NB17B_*')
                if p.is_file() and p.name != 'NB17B_04_SHA256.csv')
manifest = pd.DataFrame([
    {'file': p.name, 'bytes': p.stat().st_size,
     'sha256': hashlib.sha256(p.read_bytes()).hexdigest()}
    for p in inputs
])
manifest.to_csv(OUT_B / 'NB17B_04_SHA256.csv', index=False)

zip_path = REVIEW / 'NB17B_AUDITORIA_PARTICIONES_4664292.zip'
all_outputs = sorted(p for p in OUT_B.glob('NB17B_*') if p.is_file())
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for p in all_outputs:
        z.write(p, arcname=f'NB17B_4664292/{p.name}')
with zipfile.ZipFile(zip_path, 'r') as z:
    assert z.testzip() is None, 'ZIP con error de integridad'
    assert len(z.namelist()) == len(all_outputs), 'Faltan archivos dentro del ZIP'

print('\n✅ Resultados guardados en My Drive:', OUT_B)
print('✅ ZIP en My Drive:', zip_path)
print('✅ Archivos incluidos:', len(all_outputs))
print('✅ SHA256 ZIP:', hashlib.sha256(zip_path.read_bytes()).hexdigest())
print('📤 Archivo que debes adjuntar aquí:', zip_path.name)
colab_files.download(str(zip_path))
